# LCA Lasers4MaaS - Pre and Post processes



In [119]:
import bw2data as bd
import bw2io as bi
from bw2io import BW2Package
from bw2calc import LCA
from pathlib import Path
import bw2analyzer as ba
import pandas as pd
from IPython.display import display


### Constants: PROJECT_NAME


In [120]:
PROJECT_NAME = "Lasers4MaaS"
ECOINVENT_DATABASE_NAME = "ecoinvent-3.12-cutoff"

bd.projects.set_current(PROJECT_NAME)
if ECOINVENT_DATABASE_NAME not in bd.databases:
    bi.import_ecoinvent_release(version="3.12", system_model="cutoff")
else:
    print(f"Database {ECOINVENT_DATABASE_NAME} already exists, skipping import.")

Database ecoinvent-3.12-cutoff already exists, skipping import.


### Variables: ei


In [121]:
ei = bd.Database(ECOINVENT_DATABASE_NAME)

### Variables: bio


In [122]:
bio = bd.Database("ecoinvent-3.12-biosphere")

### Function: find_ecoinvent_activity


In [123]:
def find_ecoinvent_activity(label, search_terms, exclusion_terms=None, locations=None):
    """Return one ecoinvent activity or issue a warning if selection is ambiguous."""
    
    matches = [
        act for act in ei
        if label.lower() in act.get('name').lower()
    ]

    if not matches:
        matches = list(ei.search(" ".join(search_terms)))
        matches = [activity for activity in matches if all(term in activity.get("name") for term in search_terms)]
        if exclusion_terms is not None:
            matches = [activity for activity in matches if all(term not in activity.get("name") for term in exclusion_terms)]

    if locations is not None:
        matches = [activity for activity in matches if all(region in activity.get("location") for region in locations)]
        
    if not matches:
        matches = list(ei.search(" ".join(search_terms)))
        matches = [activity for activity in matches if any(term in activity.get("name") for term in search_terms)]
            
    if locations is not None:
        for location in locations:
            location_matches = [
                activity for activity in matches
                if activity.get("location") == location
            ]
            
            if len(location_matches) == 1:
                return location_matches[0]
            if len(location_matches) > 1:
                # warnings.warn(
                #     f"Multiple activities match {label!r} in {location!r}: "
                #     f"{location_matches}"
                # )
                return location_matches

    # If exactly one match remains, return it
    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        return matches

    return None

### Call: find_ecoinvent_activity


In [124]:
find_ecoinvent_activity(
    "abrasive blasting, alumina, carbon steel substrate", ("aluminium oxide", "production")
)


['abrasive blasting, alumina, carbon steel substrate' (square meter, GLO, None),
 'market for abrasive blasting, alumina, carbon steel substrate' (square meter, GLO, None)]

### Global parameters

In [125]:
# Technosphere exchanges
# ----------------

# ENERGY
electricity = find_ecoinvent_activity(
    "market for electricity", ("market for electricity", "medium voltage"), locations=["RER"]
)

# WATER
water = find_ecoinvent_activity(
    "tap water", ("market for tap water"), ("treatment"), locations=["RER"]
)

# CHEMICALS
chemical_alkaline = find_ecoinvent_activity(
    "sodium hydroxide, without water, in 50% solution state", ("sodium hydroxide 50% solution state"), locations=["RER"]
)
acid_pickling = find_ecoinvent_activity(
    "hydrochloric acid, without water, in 30% solution state", ("hydrochloric acid", "30% solution state", "production"), locations=["RER"]
)
nitric_acid = find_ecoinvent_activity(
    "nitric acid, without water, in 50% solution state", ("nitric acid", "50% solution state", "production"), locations=["RER"]
)

hydrochloric_acid = find_ecoinvent_activity(
    "hydrochloric acid, without water, in 30% solution state", ("hydrochloric acid", "30% solution state", "production"), locations=["RER"])

### ABRASIVES & CONSUMABLES


In [126]:
# ABRASIVES & CONSUMABLES
abrasive = find_ecoinvent_activity(
    "market for abrasive blasting, alumina, carbon steel substrate", ("aluminium oxide", "production")
)
grinding_wheel = find_ecoinvent_activity(
    "grinding wheel", ("grinding wheel", "conventional", "production"), locations=["RER"]
)
polishing_compound = find_ecoinvent_activity(
    "aluminium oxide polishing", ("aluminium oxide", "production"), locations=["RER"]
)

# COOLANTS
coolant = find_ecoinvent_activity(
    "metal working fluid", ("metal working fluid", "production"), locations=["RER"]
)

### WASTE MANAGEMENT


In [127]:

# WASTE MANAGEMENT
waste_treatment = find_ecoinvent_activity(
    "waste steel landfill", ("treatment", "waste steel", "inert material landfill"), locations=["RER"]
)
waste_paint_treatment = find_ecoinvent_activity(
    "waste paint incineration", ("treatment", "waste paint", "hazardous waste incineration", "with energy recovery"), locations=["RER"]
)
waste_spent_solvent_treatment = find_ecoinvent_activity(
    "spent solvent incineration", ("treatment", "spent solvent mixture", "hazardous waste incineration"), locations=["RER"]
)
waste_spent_solvent_energy_recovery = find_ecoinvent_activity(
    "spent solvent energy recovery", ("treatment", "spent solvent mixture", "hazardous waste incineration", "with energy recovery"), locations=["RER"]
)

### Cost factors for the main resource inputs (adjust as needed)


In [128]:
# Cost factors for the main resource inputs (adjust as needed)
COST_FACTORS = {
    'electricity_EUR_per_kWh': 0.29,     # EUR 0.29/kWh for industrial electricity
    'argon_EUR_per_kg': 5.87,           # EUR 2.5/kg for argon gas
    'nitrogen_EUR_per_kg': 5.35,          # EUR 1.2/kg for nitrogen gas

    # Consumables of the pre-/post-joining processes. INDICATIVE PLACEHOLDER PRICES:
    # replace with supplier quotes. Chemicals are per kg of ACTIVE substance, as modelled
    # (ecoinvent "without water" datasets), so they are higher than the diluted commercial price.
    'water_EUR_per_kg': 0.004,           # tap water incl. discharge, ~4 EUR/m3
    'naoh_EUR_per_kg': 0.50,             # sodium hydroxide
    'hcl_EUR_per_kg': 0.25,              # hydrochloric acid
    'hno3_EUR_per_kg': 0.45,             # nitric acid
    'alumina_grit_EUR_per_kg': 1.50,     # abrasive grit / polishing compound (alumina proxy)
    'silicon_carbide_EUR_per_kg': 2.50,  # grinding wheel (silicon carbide proxy)
    'lubricating_oil_EUR_per_kg': 2.00,  # coolant (lubricating oil proxy)
    'tungsten_carbide_EUR_per_kg': 60.0, # cutting tool material
}

### Default industrial tank sizes (liters) for chemical processes


In [129]:
# Default industrial tank sizes (liters) for chemical processes
INDUSTRIAL_TANK_SIZES = {
    'pickling': 10000,           # 10 m³ - standard for joining prep (ASTM A380)
    'alkaline_degreasing': 7500, # 7.5 m³ - common for degreasing
    'passivation': 7500,          # 7.5 m³ - similar to degreasing
    'chemical_etching': 7500,    # 7.5 m³ - similar to pickling
    'conversion_coating': 3000,  # 3 m³ - smaller for dip coating
}

### Get only the most used EF 3.1 methods


In [130]:
# Get only the most used EF 3.1 methods
most_used_indicators = [
    'climate change: total',
    'particulate matter formation',
    'acidification',
    'eutrophication: terrestrial',
    'photochemical oxidant formation: human health',
    'human toxicity: carcinogenic'
]

ef31_methods = [
    m for m in bd.methods
    if ('EF v3.1' in str(m) or 'EF 3.1' in str(m))
    and 'no LT' not in str(m)
    and any(indicator in str(m) for indicator in most_used_indicators)
]

print(f"Found {len(ef31_methods)} most-used EF 3.1 methods")
print("Selected EF 3.1 impact categories:")
for m in ef31_methods:
    print(f"  {m}")

Found 8 most-used EF 3.1 methods
Selected EF 3.1 impact categories:
  ('ecoinvent-3.12', 'EF v3.1', 'acidification', 'accumulated exceedance (AE)')
  ('ecoinvent-3.12', 'EF v3.1', 'eutrophication: terrestrial', 'accumulated exceedance (AE)')
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic', 'comparative toxic unit for human (CTUh)')
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, inorganics', 'comparative toxic unit for human (CTUh)')
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, organics', 'comparative toxic unit for human (CTUh)')
  ('ecoinvent-3.12', 'EF v3.1', 'particulate matter formation', 'impact on human health')
  ('ecoinvent-3.12', 'EF v3.1', 'photochemical oxidant formation: human health', 'tropospheric ozone concentration increase')
  ('ecoinvent-3.12', 'EN15804+A2 - Core impact categories and indicators', 'climate change: total (EF v3.1 - IPCC 2021)', 'global warming potential (GWP100)')


### Calculation engine for pre and post processes

In [131]:
# flow variable -> COST_FACTORS key. The flows are defined further down, so they are
# resolved when a cost is calculated (the mechanical proxies in mech_flows take precedence).
FLOW_COST_KEYS = {
    "electricity": "electricity_EUR_per_kWh",
    "water": "water_EUR_per_kg",
    "chemical_alkaline": "naoh_EUR_per_kg",
    "acid_pickling": "hcl_EUR_per_kg",
    "hydrochloric_acid": "hcl_EUR_per_kg",
    "nitric_acid": "hno3_EUR_per_kg",
    "abrasive": "alumina_grit_EUR_per_kg",
    "polishing_compound": "alumina_grit_EUR_per_kg",
    "grinding_wheel": "silicon_carbide_EUR_per_kg",
    "coolant": "lubricating_oil_EUR_per_kg",
    "tool_material": "tungsten_carbide_EUR_per_kg",
}


def flow_price_lookup(cost_factors=None):
    """Map ecoinvent activity key -> price (EUR per ecoinvent unit of that flow)."""
    if cost_factors is None:
        cost_factors = COST_FACTORS
    flows = {**{name: globals().get(name) for name in FLOW_COST_KEYS}, **globals().get("mech_flows", {})}
    return {
        flows[name].key: cost_factors[cost_key]
        for name, cost_key in FLOW_COST_KEYS.items()
        if hasattr(flows.get(name), "key") and cost_key in cost_factors
    }


def calculate_process_cost(process, scenario="amount", cost_factors=None):
    """Calculate consumable cost (EUR) from exchange amounts.

    Electricity, chemicals, water, abrasives, coolant and tool material are priced
    through COST_FACTORS; technosphere inputs without a price are reported once.
    """
    prices = flow_price_lookup(cost_factors)

    total_cost = 0.0
    for exchange in process.exchanges():
        if exchange["type"] == "technosphere":
            # Get amount for chosen scenario
            if scenario == "minimum":
                amount = exchange.get("minimum", exchange.get("min", exchange["amount"]))
            elif scenario == "maximum":
                amount = exchange.get("maximum", exchange.get("max", exchange["amount"]))
            else:
                amount = exchange["amount"]

            price = prices.get(exchange.input.key)
            if price is None:
                print(f"WARNING {process['name']}: no price for {exchange.input['name']!r}, not costed")
                continue
            total_cost += amount * price
    return total_cost

### Common Uvalues (W/m²·K)


In [132]:
# ======================
# Common U-values (W/m²·K)
# ======================
U_BARE_STEEL = 9.8      # Uninsulated steel tank
U_STAINLESS_STEEL = 8.5
U_FIBERGLASS_5CM = 0.57 # 5 cm fiberglass insulation
U_POLYURETHANE_5CM = 0.45
U_TYPICAL_INSULATED = 0.6

U_VALUES = {
    "bare_steel": U_BARE_STEEL,
    "stainless_steel": U_STAINLESS_STEEL,
    "fiberglass_5cm": U_FIBERGLASS_5CM,
    "polyurethane_5cm": U_POLYURETHANE_5CM,
    "typical_insulated": U_TYPICAL_INSULATED
}

def tank_heat_loss(volume, fluid_temp, room_temp, u_value, time_hours=1):
    """
    Calculate heat loss for a cubic fluid tank.

    Args:
        volume (m³): Tank volume
        fluid_temp (°C): fluid temperature
        room_temp (°C): Room temperature
        u_value (W/m²·K): Heat transfer coefficient
        time_hours (h): Time duration (default: 1)

    Returns:
        (heat_loss_kWh, temp_drop_°C, final_temp_°C)
    """
    side = volume ** (1/3)
    area = 6 * side**2
    mass = volume * 1000
    delta_t = fluid_temp - room_temp
    heat_loss_kwh = u_value * area * delta_t * time_hours / 1000
    temp_drop = heat_loss_kwh / (mass * 4186)
    heat_maintenance_kwh = heat_loss_kwh / 0.95 # 95% heat element efficiency
    return heat_loss_kwh, temp_drop, fluid_temp - temp_drop, heat_maintenance_kwh

## Chemical Pre and Post processes

In [ ]:
CHEM_PREPOST_NAME = "Chemical_Pre_and_Post_processes"
CHEM_PREPOST = bd.Database(CHEM_PREPOST_NAME)

# Pre-/post-joining process parameters with min/max ranges.
# Each process appears once; material-specific fields are keyed by "steel" / "aluminium".
# Fields identical for both materials (e.g. description) are stored as plain values.
CHEM_PREPOST_PARAMS = {
"alkaline_degreasing": {
            "standard": {
                "steel": "ASTM B183",
                "aluminium": "ASTM B254",
            },
            "parameters": {
                "steel": {
                    "acid_concentration_%": {"min": 0.05, "max": 0.1},
                    "water_consumption_l": {"min": 5.0, "max": 15.0},
                    "temperature_C": {"min": 60, "max": 80},
                    "time_min": {"min": 5, "max": 15},
                },
                "aluminium": {
                    "acid_concentration_%": {"min": 0.03, "max": 0.1},
                    "water_consumption_l": {"min": 3.0, "max": 10.0},
                    "temperature_C": {"min": 50, "max": 70},
                    "time_min": {"min": 8, "max": 20},
                },
            },
            "description": {
                "steel": "Remove oils, grease, and contaminants from steel surface",
                "aluminium": "Remove oxides and contaminants using alkaline solutions with inhibitors",
            },
        },
"chemical_etching": {
            "standard": "ASTM B253",
            "parameters": {
                "aluminium": {
                    "acid_concentration_%": {"min": 0.05, "max": 0.10},
                    "water_consumption_l": {"min": 5.0, "max": 15.0},
                    "time_min": {"min": 2, "max": 10},
                    "temperature_C": {"min": 50, "max": 65},
                },
            },
            "description": "Alkaline (NaOH) etch of aluminium to remove the natural oxide layer and a thin surface layer",
        },
"pickling": {
            "standard": "ASTM A380, ISO 16048",
            "parameters": {
                "steel": {
                    "acid_concentration_%": {"min": .1, "max": 0.2},
                    "hydrofluoric_acid_consumption_%": {"min": 0.01, "max": 0.02},
                    "water_consumption_l": {"min": 10.0, "max": 25.0},
                    "time_min": {"min": 10, "max": 30},
                    "temperature_C": {"min": 20, "max": 50},
                },
            },
            "description": "Remove weld scale and oxides using acid solutions",
        },
"passivation": {
            "standard": "ASTM A380, ASTM A967",
            "parameters": {
                "steel": {
                    "acid_concentration_%": {"min": 0.2, "max": 0.4},
                    "water_consumption_l": {"min": 5.0, "max": 10.0},
                    "time_min": {"min": 20, "max": 30},
                    "temperature_C": {"min": 20, "max": 50},
                },
            },
            "description": "Form protective oxide layer on stainless steel surface",
        },
}

### Function: create_chemical_process_exchanges


In [134]:
BATH_DENSITY_KG_PER_L = 1.0  # assumed density of the dilute process baths (water-like)


def create_chemical_process_exchanges(
    database_name,
    process_params,
    process_name,
    solution_volume_l=None,
    service_name=None,
):
    """Build Brightway exchanges from CHEM_PREPOST_PARAMS.

    `process_name` is the unique process key (service + material); `service_name`
    (default: `process_name`) selects the chemical and the default tank size.
    `solution_volume_l` is a number in litres (default: INDUSTRIAL_TANK_SIZES).
    `acid_concentration_%` values are mass fractions of active chemical in the
    bath: 0.05 means 5 wt%. Chemical inputs are kg of active substance, matching
    the ecoinvent "without water" datasets. Bath density is BATH_DENSITY_KG_PER_L.
    """
    service_name = service_name or process_name

    default_flows = {
        "water": globals().get("water"),
        "chemical_alkaline": globals().get("chemical_alkaline"),
        "nitric_acid": globals().get("nitric_acid"),
        "electricity": globals().get("electricity"),
        "acid_pickling": globals().get("acid_pickling"),
        "hydrochloric_acid": globals().get("hydrochloric_acid")
    }

    def as_key(activity_or_key):
        return getattr(activity_or_key, "key", activity_or_key)

    def bounds(value):
        if isinstance(value, dict):
            return float(value["min"]), float(value["max"])
        value = float(value)
        return value, value

    exchanges = [
            {
                "input": (database_name, process_name),
                "amount": 1,
                "type": "production",
            }
        ]

    acid = {
        "pickling": "acid_pickling",
        "alkaline_degreasing": "chemical_alkaline",
        "passivation": "nitric_acid",
        "chemical_etching": "chemical_alkaline",
    }[service_name]

    if solution_volume_l is None:
        solution_volume_l = INDUSTRIAL_TANK_SIZES.get(service_name, 7500)  # default 7.5 m3

    time_min, time_max = bounds(process_params["time_min"])
        
    volume_min, volume_max = bounds(solution_volume_l)
    
    replenishment_factor = 0.001 # fraction of the bath's active chemical replenished per hour (losses, degradation)

    # acid mass (kg of active substance): wt fraction * bath mass * replenishment per hour * hours
    concentration_min, concentration_max = bounds(process_params["acid_concentration_%"])
    acid_min = concentration_min * volume_min * BATH_DENSITY_KG_PER_L * replenishment_factor * time_min / 60
    acid_max = concentration_max * volume_max * BATH_DENSITY_KG_PER_L * replenishment_factor * time_max / 60
    exchanges.append({
        "input": as_key(default_flows[acid]),
        "amount": (acid_max + acid_min)/2,
        "type": "technosphere",
        "minimum": acid_min,
        "maximum": acid_max,
        "uncertainty_type": 1,
    })


    # electricity for heat maintenance
    temp_min, temp_max = bounds(process_params["temperature_C"])
    electricity_kwh_min = tank_heat_loss(solution_volume_l/1000, temp_min, room_temp=20, u_value=U_VALUES["typical_insulated"], time_hours=time_min/60)
    electricity_kwh_max = tank_heat_loss(solution_volume_l/1000, temp_max, room_temp=20, u_value=U_VALUES["typical_insulated"], time_hours=time_max/60)
    exchanges.append({
        "input": as_key(default_flows["electricity"]),
        "amount":(electricity_kwh_max[3] + electricity_kwh_min[3])/2,
        "type": "technosphere",
        "minimum": electricity_kwh_min[3],
        "maximum": electricity_kwh_max[3],
        "uncertainty_type": 1,
    })

    # ionized water consumption rinsing
    water_consumption_min, water_consumption_max = bounds(process_params["water_consumption_l"]) 
    
    exchanges.append({
        "input": as_key(default_flows["water"]),
        "amount": (water_consumption_max + water_consumption_min)/2,
        "type": "technosphere",
        "minimum": water_consumption_min,
        "maximum": water_consumption_max,
        "uncertainty_type": 1,
    })
    return exchanges


ChemProcess_data = {}

for service_name in CHEM_PREPOST_PARAMS:
    record = CHEM_PREPOST_PARAMS.get(service_name)
    materials = list(record.get("parameters", {}).keys())
    for material in materials:
        process_by_material_data = record.get("parameters", {}).get(material)
        process_key = f"{service_name}_{material}"  # one process per service AND material
        ChemProcess_data[(CHEM_PREPOST_NAME, process_key)] = {
            "name": f"{service_name} for {material}",
            "unit": "unit",
            "location": "NL",
            "exchanges": create_chemical_process_exchanges(
                CHEM_PREPOST_NAME,
                process_by_material_data,
                process_key,
                service_name=service_name,
            ),
        }
    

### Call: CHEM_PREPOST.write


In [135]:
CHEM_PREPOST.write(ChemProcess_data)

100%|██████████| 5/5 [00:00<00:00, 84904.94it/s]

13:17:08+0100 [info     ] Vacuuming database            


## Mechanical pre and post processes

In [136]:
MECH_PREPOST_NAME = "Mechanical_Pre_and_Post_processes"
MECH_PREPOST = bd.Database(MECH_PREPOST_NAME)

# Mechanical pre-/post-joining treatments.
MECH_PREPOST_PARAMS = {
"mechanical_treatment": {
            "standard": {
                "steel": "ISO 8501-1 (surface preparation)",
                "aluminium": "ISO 8501-1",
            },
            "parameters": {
                "steel": {
                    "abrasive_consumption_kg": {"min": 0.01, "max": 0.05},
                    "electricity_kwh": {"min": 0.1, "max": 0.4},
                    "time_min": {"min": 5, "max": 20},
                    "surface_area_m2": {"min": 0.5, "max": 2.0},
                },
                "aluminium": {
                    "abrasive_consumption_kg": {"min": 0.005, "max": 0.02},
                    "coolant_consumption_l": {"min": 0.5, "max": 2.0},
                    "electricity_kwh": {"min": 0.05, "max": 0.2},
                    "time_min": {"min": 3, "max": 15},
                    "surface_area_m2": {"min": 0.3, "max": 1.5},
                },
            },
            "condition": {
                "steel": "if oxides/scale are present",
            },
            "note": {
                "aluminium": "Use non-ferrous tools with coolant",
            },
            "description": {
                "steel": "Remove oxides and scale using abrasive methods",
                "aluminium": "Surface preparation using non-ferrous tools with mandatory coolant",
            },
        },
"machining": {
            "standard": "ISO 3685 (tool life)",
            "parameters": {
                "steel": {
                    "machining_time_min": {"min": 1, "max": 5},
                    "tool_wear_pct": {"min": 5, "max": 15},
                    "electricity_kwh": {"min": 0.05, "max": 0.2},
                },
            },
            "condition": "if tight tolerances are required",
            "description": "Precision machining for tight tolerance requirements",
        },
"grinding_polishing":{
            "standard": {
                "steel": "ISO 16048, ASTM B912 (electropolishing)",
                "aluminium": "ISO 16048, ASTM B209",
            },
            "parameters": {
                "steel": {
                    "grinding_wheel_consumption_kg": {"min": 0.005, "max": 0.02},
                    "polishing_compound_kg": {"min": 0.01, "max": 0.05},
                    "electricity_kwh": {"min": 0.1, "max": 0.5},
                    "time_min": {"min": 5, "max": 25},
                    "surface_roughness_um": {"min": 0.1, "max": 1.6},
                },
                "aluminium": {
                    "grinding_wheel_consumption_kg": {"min": 0.002, "max": 0.01},
                    "polishing_compound_kg": {"min": 0.005, "max": 0.02},
                    "coolant_consumption_l": {"min": 0.1, "max": 0.5},
                    "electricity_kwh": {"min": 0.05, "max": 0.2},
                    "time_min": {"min": 3, "max": 15},
                    "surface_roughness_um": {"min": 0.05, "max": 1.0},
                },
            },
            "condition": {
                "steel": "as needed for surface finish",
            },
            "note": {
                "aluminium": "Use non-ferrous tools with mandatory coolant",
            },
            "description": {
                "steel": "Achieve required surface finish and dimensional accuracy",
                "aluminium": "Final surface finishing using non-ferrous tools with coolant",
            },
        }
}

### Function: create_mechanical_process_exchanges


In [137]:
# ---- Mechanical-process flows -------------------------------------------------
# ecoinvent 3.12 has NO "grinding wheel" or "metal working fluid" dataset, and
# "abrasive blasting, alumina" is a service in m2 (not kg of abrasive). The
# flows below are therefore PROXIES, all in kg. Review/replace as needed.
def _pick(name, location, reference_product=None):
    """Exact-name ecoinvent lookup; fails loudly instead of returning a list."""
    hits = [
        a for a in ei
        if a["name"] == name and a["location"] == location
        and (reference_product is None or a.get("reference product") == reference_product)
    ]
    if len(hits) != 1:
        raise ValueError(f"{name!r} [{location}] gave {len(hits)} matches; refine the lookup")
    return hits[0]

mech_flows = {
    "electricity": electricity,                                   # kWh
    # abrasive grit (kg): alumina grit proxy (corundum), non-metallurgical grade
    "abrasive": _pick("aluminium oxide production", "RoW", "aluminium oxide, non-metallurgical"),
    # grinding wheel (kg): silicon carbide proxy (abrasive grain of wheels)
    "grinding_wheel": _pick("market for silicon carbide", "GLO"),
    # polishing compound (kg): alumina polishing powder proxy
    "polishing_compound": _pick("aluminium oxide production", "RoW", "aluminium oxide, non-metallurgical"),
    # coolant / metalworking fluid (kg): lubricating oil proxy (no emulsion dataset in ecoinvent)
    "coolant": _pick("market for lubricating oil", "RER"),
    # cutting tool material (kg): tungsten carbide powder proxy
    "tool_material": _pick("market for tungsten carbide powder", "GLO"),
}

# ---- Assumptions needed to convert the parameters to ecoinvent units ----------
COOLANT_DENSITY_KG_PER_L = 0.9   # assumed density of coolant
TOOL_MASS_KG = 0.01              # rule of thumb: ~10 g carbide indexable insert (typical 5-15 g),
                                 # consumed at 100 % wear; set to None to skip tool wear


def create_mechanical_process_exchanges(
    database_name,
    process_params,
    process_name,
    tool_mass_kg=None,
    coolant_density_kg_per_l=None,
):
    """Build Brightway exchanges from MECH_PREPOST_PARAMS.

    `process_params` is the material-specific dict (e.g. ["parameters"]["steel"]).
    Each parameter is a {"min", "max"} range (or a number); the exchange amount
    is the midpoint, with "minimum"/"maximum" kept for the scenario runs.
    Amounts are the totals consumed over one execution of the process (its
    duration, time_min), so the functional unit is 1 execution and no
    time scaling is applied.
    Parameters without an ecoinvent flow (time, area, roughness, ...) are ignored.
    """
    tool_mass_kg = TOOL_MASS_KG if tool_mass_kg is None else tool_mass_kg
    coolant_density_kg_per_l = COOLANT_DENSITY_KG_PER_L if coolant_density_kg_per_l is None else coolant_density_kg_per_l

    def as_key(activity_or_key):
        return getattr(activity_or_key, "key", activity_or_key)

    def bounds(value):
        if isinstance(value, dict):
            return float(value["min"]), float(value["max"])
        value = float(value)
        return value, value

    # parameter -> (flow, factor to the ecoinvent unit)
    parameter_map = {
        "electricity_kwh": ("electricity", 1.0),
        "abrasive_consumption_kg": ("abrasive", 1.0),
        "grinding_wheel_consumption_kg": ("grinding_wheel", 1.0),
        "polishing_compound_kg": ("polishing_compound", 1.0),
        "coolant_consumption_l": ("coolant", coolant_density_kg_per_l),
    }
    if tool_mass_kg is not None:
        parameter_map["tool_wear_pct"] = ("tool_material", tool_mass_kg / 100)  # % -> kg

    exchanges = [
        {
            "input": (database_name, process_name),
            "amount": 1,
            "type": "production",
        }
    ]

    ignored = []
    for parameter, value in process_params.items():
        if parameter not in parameter_map:
            ignored.append(parameter)
            continue
        flow_name, factor = parameter_map[parameter]
        low, high = (b * factor for b in bounds(value))
        exchanges.append({
            "input": as_key(mech_flows[flow_name]),
            "amount": (low + high) / 2,
            "type": "technosphere",
            "minimum": low,
            "maximum": high,
            "uncertainty_type": 1,
        })

    if tool_mass_kg is None and "tool_wear_pct" in ignored:
        print(f"WARNING {process_name}: tool_wear_pct skipped; set TOOL_MASS_KG to include tool wear")
    print(f"{process_name}: ignored (no ecoinvent flow): {ignored}")
    return exchanges


In [138]:
MechProcess_data = {}

for service_name, record in MECH_PREPOST_PARAMS.items():
    for material, material_params in record["parameters"].items():
        process_key = f"{service_name}_{material}"  # one process per service AND material
        MechProcess_data[(MECH_PREPOST_NAME, process_key)] = {
            "name": f"{service_name} for {material}",
            "unit": "unit",
            "location": "NL",
            "exchanges": create_mechanical_process_exchanges(
                MECH_PREPOST_NAME,
                material_params,
                process_key,
            ),
        }

MECH_PREPOST.write(MechProcess_data)  # uncomment once the proxies/assumptions above are confirmed


mechanical_treatment_steel: ignored (no ecoinvent flow): ['time_min', 'surface_area_m2']
mechanical_treatment_aluminium: ignored (no ecoinvent flow): ['time_min', 'surface_area_m2']
machining_steel: ignored (no ecoinvent flow): ['machining_time_min']
grinding_polishing_steel: ignored (no ecoinvent flow): ['time_min', 'surface_roughness_um']
grinding_polishing_aluminium: ignored (no ecoinvent flow): ['time_min', 'surface_roughness_um']


100%|██████████| 5/5 [00:00<00:00, 75983.77it/s]

13:17:24+0100 [info     ] Vacuuming database            


## LCIA results

In [139]:
import hashlib
import inspect
import json
import pickle


def _compute_lcia_scenarios(process_databases, ef31_methods, cost_factors):
    scenario_results = []
    category_units = {}

    scenarios = ["minimum", "amount", "maximum"]
    scenario_keys = {
        "minimum": ["minimum", "min"],
        "amount": ["amount"],
        "maximum": ["maximum", "max"]
    }

    for database in process_databases:
        for process in database:
            process_exchanges = list(process.exchanges())

            for scenario in scenarios:
                original_amounts = [
                    (exchange, exchange["amount"])
                    for exchange in process_exchanges
                ]

                try:
                    for exchange, original_amount in original_amounts:
                        scenario_amount = original_amount
                        for key in scenario_keys[scenario]:
                            if key in exchange:
                                scenario_amount = exchange[key]
                                break

                        if scenario_amount != original_amount:
                            exchange["amount"] = scenario_amount
                            exchange.save()

                    # Calculate LCIA
                    lca = LCA({process: 1}, method=ef31_methods[0])
                    lca.lci()
                    lca.lcia()

                    # Calculate foreground electricity consumption and total cost
                    electricity_kwh = sum(
                        exchange["amount"]
                        for exchange in process_exchanges
                        if exchange["type"] == "technosphere"
                        and "electricity" in str(exchange.input).lower()
                    )
                    opex = calculate_process_cost(process, scenario, cost_factors)

                    for method in ef31_methods:
                        lca.switch_method(method)
                        lca.lcia()

                        method_tuple = str(method).replace("'", '').strip("()")
                        method_parts = [p.strip() for p in method_tuple.split(",")]
                        impact_category = method_parts[2] if len(method_parts) > 2 else str(method)

                        category_units[impact_category] = bd.Method(method).metadata.get("unit", "")

                        scenario_results.append({
                            "Database": database.filename,
                            "Process": process["name"],
                            "Key": process.key[1],
                            "Scenario": scenario,
                            "Method": str(method),
                            "Impact Category": impact_category,
                            "Amount": lca.score,
                            "Electricity Consumption (kWh)": electricity_kwh,
                            "Consumable costs (EUR)": opex
                        })
                finally:
                    for exchange, original_amount in original_amounts:
                        if exchange["amount"] != original_amount:
                            exchange["amount"] = original_amount
                            exchange.save()

    return scenario_results, category_units


# Set to True (or pass force_recalc=True) to ignore the cache and recompute the LCIA
FORCE_RECALC = False
LCIA_CACHE_DIR = Path("results/.cache")


def _lcia_cache_key(process_databases, ef31_methods, cost_factors):
    """Hash everything the LCIA results depend on, so edits invalidate the cache."""
    parts = [ECOINVENT_DATABASE_NAME, repr(sorted(cost_factors.items())), repr(list(ef31_methods)),
             repr(sorted((repr(k), v) for k, v in flow_price_lookup(cost_factors).items()))]
    try:
        parts.append(inspect.getsource(calculate_process_cost))
    except (OSError, TypeError):
        pass
    dump = lambda obj: json.dumps(dict(obj._data), sort_keys=True, default=str)
    for database in process_databases:
        # Database iteration order is not stable, so sort before hashing
        for process in sorted(database, key=lambda a: a.key):
            parts.append(dump(process))
            parts.extend(sorted(dump(exchange) for exchange in process.exchanges()))
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


def calculate_lcia_scenarios(process_databases, ef31_methods, cost_factors=None, force_recalc=None):
    if cost_factors is None:
        cost_factors = COST_FACTORS
    if force_recalc is None:
        force_recalc = FORCE_RECALC

    if not isinstance(process_databases, (list, tuple)):
        process_databases = [process_databases]

    LCIA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = LCIA_CACHE_DIR / f"lcia_{_lcia_cache_key(process_databases, ef31_methods, cost_factors)}.pkl"

    if cache_file.exists() and not force_recalc:
        with open(cache_file, "rb") as f:
            scenario_results, category_units = pickle.load(f)
        print(f"Loaded cached LCIA results from {cache_file} (set FORCE_RECALC = True to recompute)")
    else:
        scenario_results, category_units = _compute_lcia_scenarios(process_databases, ef31_methods, cost_factors)
        with open(cache_file, "wb") as f:
            pickle.dump((scenario_results, category_units), f)
        print(f"Computed LCIA results and cached them in {cache_file}")

    # Create DataFrame and add costs
    scenario_df = pd.DataFrame(scenario_results)

    pivot_scenario_df = scenario_df.pivot_table(
        index=["Database", "Process", "Key", "Scenario"],
        columns="Impact Category",
        values="Amount",
        aggfunc="sum"
    ).reset_index()

    result_index = ["Database", "Process", "Key", "Scenario"]
    resource_df = scenario_df[result_index + ["Electricity Consumption (kWh)", "Consumable costs (EUR)"]].drop_duplicates(subset=result_index)

    final_df = pd.merge(pivot_scenario_df, resource_df,
                       on=result_index, how="left")

    # Column headers with units for display and export; final_df keeps plain names
    export_df = final_df.rename(columns={
        cat: f"{cat} [{unit}]" for cat, unit in category_units.items() if unit
    })

    print("\n\n=== EF 3.1 LCIA Results with Foreground Electricity Consumption and Costs ===")
    print(export_df.to_markdown(index=False, floatfmt=".4e"))

    # Save results as CSV and JSON
    if len(process_databases) == 1:
        db_name = process_databases[0].filename.split('.')[0]
    else:
        db_name = "_".join(db.filename.split('.')[0] for db in process_databases)

    output_file = f"results/{db_name}_LCIA_results.csv"
    json_file = f"results/{db_name}_LCIA_results.json"

    export_df.to_csv(output_file, index=False)
    export_df.to_json(json_file, orient="records", indent=2)

    print(f"\nScenario results saved to {output_file} and {json_file}")

    return final_df


### Call: calculate_lcia_scenarios


In [140]:
calculate_lcia_scenarios([CHEM_PREPOST, MECH_PREPOST], ef31_methods)

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 7.65e+12)
  warnings.warn(msg, UmfpackWarning)


Computed LCIA results and cached them in results/.cache/lcia_7619c12b35f01ec3.pkl


=== EF 3.1 LCIA Results with Foreground Electricity Consumption and Costs ===
| Database                                   | Process                            | Key                            | Scenario   |   acidification [mol H+-Eq] |   climate change: total (EF v3.1 - IPCC 2021) [kg CO2-Eq] |   eutrophication: terrestrial [mol N-Eq] |   human toxicity: carcinogenic [CTUh] |   particulate matter formation [disease incidence] |   photochemical oxidant formation: human health [kg NMVOC-Eq] |   Electricity Consumption (kWh) |   Consumable costs (EUR) |
|:-------------------------------------------|:-----------------------------------|:-------------------------------|:-----------|----------------------------:|----------------------------------------------------------:|-----------------------------------------:|--------------------------------------:|---------------------------------------------------:|--

,Database,Process,Key,Scenario,acidification,climate change: total (EF v3.1 - IPCC 2021),eutrophication: terrestrial,human toxicity: carcinogenic,particulate matter formation,photochemical oxidant formation: human health,Electricity Consumption (kWh),Consumable costs (EUR)
0,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,amount,0.000884,0.198565,0.001618,2.127498e-09,5.891000e-09,0.000495,0.150035,0.139510
1,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,maximum,0.001532,0.344737,0.002818,3.795154e-09,1.033873e-08,0.000862,0.241992,0.235178
2,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,minimum,0.000236,0.052393,0.000418,4.598414e-10,1.443271e-09,0.000129,0.058078,0.043843
3,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for steel,alkaline_degreasing_steel,amount,0.000725,0.162395,0.001320,1.666894e-09,4.786442e-09,0.000404,0.133096,0.133285
4,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for steel,alkaline_degreasing_steel,maximum,0.001223,0.273983,0.002230,2.855052e-09,8.105044e-09,0.000683,0.217793,0.216910
5,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for steel,alkaline_degreasing_steel,minimum,0.000228,0.050807,0.000410,4.787372e-10,1.467840e-09,0.000126,0.048398,0.049661
6,Chemical_Pre_and_Post_processes.9a629b55,chemical_etching for aluminium,chemical_etching_aluminium,amount,0.000425,0.095338,0.000784,1.047471e-09,2.949906e-09,0.000240,0.061708,0.092270
7,Chemical_Pre_and_Post_processes.9a629b55,chemical_etching for aluminium,chemical_etching_aluminium,maximum,0.000762,0.171154,0.001408,1.902199e-09,5.285025e-09,0.000431,0.108897,0.154080
8,Chemical_Pre_and_Post_processes.9a629b55,chemical_etching for aluminium,chemical_etching_aluminium,minimum,0.000088,0.019523,0.000161,1.927422e-10,6.147873e-10,0.000049,0.014520,0.030461
9,Chemical_Pre_and_Post_processes.9a629b55,passivation for steel,passivation_steel,amount,0.009266,1.050288,0.037359,3.361823e-10,4.630664e-08,0.002288,0.108897,0.511580


## Regional analysis: EU electricity markets
Replaces the RER electricity market with each member state's own ecoinvent market and maps the resulting impacts and consumable costs per execution of each pre-/post-joining process. Only electricity changes: water, chemicals and abrasives keep their ecoinvent datasets and prices.

In [141]:
import hashlib, pickle
import numpy as np

EU27 = {"AT":"AUT","BE":"BEL","BG":"BGR","HR":"HRV","CY":"CYP","CZ":"CZE","DK":"DNK","EE":"EST","FI":"FIN",
        "FR":"FRA","DE":"DEU","GR":"GRC","HU":"HUN","IE":"IRL","IT":"ITA","LV":"LVA","LT":"LTU","LU":"LUX",
        "MT":"MLT","NL":"NLD","PL":"POL","PT":"PRT","RO":"ROU","SK":"SVK","SI":"SVN","ES":"ESP","SE":"SWE"}

# Industrial electricity prices by country, EUR/kWh (Eurostat nrg_pc_205, 2025-S1, band ID 2-20 GWh/yr,
# excl. VAT and recoverable taxes), as in joining_task.ipynb. "RER" uses the EU27 average.
ELEC_PRICE = {"BE": 0.1659, "BG": 0.1206, "CZ": 0.1655, "DK": 0.126, "DE": 0.1932, "EE": 0.1302, "IE": 0.2357, "GR": 0.1811, "ES": 0.125, "FR": 0.1296, "HR": 0.1534, "IT": 0.203, "CY": 0.2267, "LV": 0.113, "LT": 0.1347, "LU": 0.1476, "HU": 0.1871, "MT": 0.1177, "NL": 0.1889, "AT": 0.1803, "PL": 0.1726, "PT": 0.123, "RO": 0.1633, "SI": 0.1456, "SK": 0.1782, "FI": 0.0697, "SE": 0.0838,
              "RER": 0.164}

SCENARIO_KEYS = {"minimum": ("minimum", "min"), "amount": ("amount",), "maximum": ("maximum", "max")}
prepost_dbs = (CHEM_PREPOST, MECH_PREPOST)

def unit_impacts(act):
    lca = LCA({act: 1}, method=ef31_methods[0]); lca.lci()
    out = {}
    for m in ef31_methods:
        lca.switch_method(m); lca.lcia(); out[m[2]] = lca.score
    return out

def exchange_amount(exchange, scenario):
    return next((exchange[k] for k in SCENARIO_KEYS[scenario] if k in exchange), exchange["amount"])

# Foreground processes only have technosphere inputs, so the score is linear in each of them
other_inputs = {e.input.key: e.input for db in prepost_dbs for p in db for e in p.technosphere()
                if e.input.key != electricity.key}

def _compute_grid():
    grid = {"RER": unit_impacts(electricity)}   # baseline used above
    for iso2 in EU27:
        market = next(a for a in ei if a["name"] == "market for electricity, medium voltage" and a["location"] == iso2)
        grid[iso2] = unit_impacts(market)
    return grid, {k: unit_impacts(a) for k, a in other_inputs.items()}

REGIONAL_CACHE_DIR = LCIA_CACHE_DIR
REGIONAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_key = hashlib.sha256(repr((ECOINVENT_DATABASE_NAME, electricity.key, sorted(other_inputs), sorted(EU27), list(ef31_methods))).encode()).hexdigest()[:16]
_cache = REGIONAL_CACHE_DIR / f"prepost_grid_{_key}.pkl"
if _cache.exists() and not FORCE_RECALC:
    with open(_cache, "rb") as f:
        grid, input_imp = pickle.load(f)
    print(f"Loaded cached grid impacts from {_cache} (set FORCE_RECALC = True to recompute)")
else:
    grid, input_imp = _compute_grid()
    with open(_cache, "wb") as f:
        pickle.dump((grid, input_imp), f)
    print(f"Computed grid impacts and cached them in {_cache}")

prices = flow_price_lookup()
rows = []
for db in prepost_dbs:
    for proc in db:
        for scenario in SCENARIO_KEYS:
            amounts = [(e.input.key, exchange_amount(e, scenario)) for e in proc.technosphere()]
            kwh = sum(a for k, a in amounts if k == electricity.key)
            others = [(k, a) for k, a in amounts if k != electricity.key]
            other_cost = sum(a * prices.get(k, 0.0) for k, a in others)
            for c, imp in grid.items():
                for ind in imp:
                    rows.append({"process": proc["name"], "key": proc.key[1], "database": db.name, "scenario": scenario, "country": c,
                                 "indicator": ind, "impact": kwh * imp[ind] + sum(a * input_imp[k][ind] for k, a in others)})
                # Consumable costs: country-specific electricity price + the other consumables at COST_FACTORS
                rows.append({"process": proc["name"], "key": proc.key[1], "database": db.name, "scenario": scenario, "country": c,
                             "indicator": "Consumable costs (EUR)", "impact": kwh * ELEC_PRICE[c] + other_cost})
regional_df = pd.DataFrame(rows)
regional_df.to_csv("results/regional_prepost.csv", index=False)   # read by workflow_assembly.ipynb
regional_df.head()

Loaded cached grid impacts from results/.cache/prepost_grid_e1067b8557ff9f9d.pkl (set FORCE_RECALC = True to recompute)


,process,key,database,scenario,country,indicator,impact
0,passivation for steel,passivation_steel,Chemical_Pre_and_Post_processes,minimum,RER,acidification,4.542986e-03
1,passivation for steel,passivation_steel,Chemical_Pre_and_Post_processes,minimum,RER,eutrophication: terrestrial,1.854307e-02
2,passivation for steel,passivation_steel,Chemical_Pre_and_Post_processes,minimum,RER,human toxicity: carcinogenic,8.128978e-11
3,passivation for steel,passivation_steel,Chemical_Pre_and_Post_processes,minimum,RER,"human toxicity: carcinogenic, inorganics",4.033078e-11
4,passivation for steel,passivation_steel,Chemical_Pre_and_Post_processes,minimum,RER,"human toxicity: carcinogenic, organics",4.095900e-11


In [142]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets

method_units = {m[2]: bd.Method(m).metadata.get("unit", "") for m in ef31_methods}
method_units["Consumable costs (EUR)"] = "EUR"
ind_opts = [m[2] for m in ef31_methods] + ["Consumable costs (EUR)"]
codes = list(EU27)
iso3  = [EU27[c] for c in codes]
short = lambda s: s.split(" (EF")[0]          # drop the "(EF v3.1 - IPCC 2021)" suffix in titles

wide = widgets.Layout(width="420px")
sty  = {"description_width": "90px"}
w_a    = widgets.Dropdown(options=ind_opts, value=ind_opts[0],  description="Indicator A", layout=wide, style=sty)
w_b    = widgets.Dropdown(options=ind_opts, value=ind_opts[-1], description="Indicator B", layout=wide, style=sty)
w_proc = widgets.Dropdown(options=sorted(regional_df.process.unique()), description="Process", layout=wide, style=sty)
w_scen = widgets.Dropdown(options=list(SCENARIO_KEYS), value="amount", description="Scenario", layout=wide, style=sty)
w_view = widgets.Dropdown(options=["Absolute", "Change vs RER (%)"], description="View", layout=wide, style=sty)

# Two maps (A, B) on top and a country-by-country scatter of A vs B below
fig = go.FigureWidget(make_subplots(
    rows=2, cols=2, specs=[[{"type": "choropleth"}, {"type": "choropleth"}], [{"type": "xy", "colspan": 2}, None]],
    row_heights=[0.6, 0.4], vertical_spacing=0.1, horizontal_spacing=0.04, subplot_titles=("A", "B", "")))
cbar = lambda x: dict(x=x, y=0.79, len=0.42, thickness=12, outlinewidth=0, tickfont=dict(size=10), title=dict(font=dict(size=11)))
hover = "<b>%{text}</b><br>%{z:.4g}<extra></extra>"
fig.add_trace(go.Choropleth(locations=iso3, z=[0] * len(codes), text=codes, hovertemplate=hover, colorbar=cbar(0.455),
                            marker=dict(line=dict(color="white", width=0.5))), row=1, col=1)
fig.add_trace(go.Choropleth(locations=iso3, z=[0] * len(codes), text=codes, hovertemplate=hover, colorbar=cbar(1.0),
                            marker=dict(line=dict(color="white", width=0.5))), row=1, col=2)
fig.add_trace(go.Scatter(x=[0] * len(codes), y=[0] * len(codes), mode="markers+text", text=codes, textposition="top center",
                         textfont=dict(size=10), marker=dict(size=9, color="#1f77b4", line=dict(color="white", width=1)),
                         hovertemplate="<b>%{text}</b><br>A: %{x:.4g}<br>B: %{y:.4g}<extra></extra>"), row=2, col=1)
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text", text=["RER"], textposition="bottom center", textfont=dict(size=10),
                         marker=dict(size=13, symbol="diamond", color="#d62728", line=dict(color="white", width=1)),
                         hovertemplate="<b>RER baseline</b><br>A: %{x:.4g}<br>B: %{y:.4g}<extra></extra>"), row=2, col=1)
fig.update_geos(scope="europe", fitbounds="locations", showframe=False, showcountries=True,
                countrycolor="#c4cad0", showland=True, landcolor="#eceff1", showocean=False, showlakes=False,
                bgcolor="rgba(0,0,0,0)")
fig.update_layout(template="plotly_white", height=950, showlegend=False, font=dict(size=12),
                  margin=dict(l=60, r=30, t=60, b=50))
fig.update_annotations(font_size=13)
fig.update_xaxes(zeroline=False, gridcolor="#e6e9ec"); fig.update_yaxes(zeroline=False, gridcolor="#e6e9ec")

def series(ind):
    d = regional_df[(regional_df.process == w_proc.value) & (regional_df.scenario == w_scen.value) & (regional_df.indicator == ind)]
    d = d.set_index("country")["impact"]
    if w_view.value == "Absolute":
        return d[codes].values, d["RER"], method_units.get(ind, "")
    return ((d[codes] / d["RER"] - 1) * 100).values, 0.0, "% vs RER"

def update(_=None):
    (za, ra, ua), (zb, rb, ub) = series(w_a.value), series(w_b.value)
    pct = w_view.value != "Absolute"
    with fig.batch_update():
        for trace, z, u, scale in ((fig.data[0], za, ua, "YlOrRd"), (fig.data[1], zb, ub, "GnBu")):
            m = max(abs(z).max(), 1e-12)
            trace.z = z
            trace.colorscale = "RdBu_r" if pct else scale
            trace.zmin, trace.zmax = (-m, m) if pct else (None, None)
            trace.colorbar.title.text = u
        fig.data[2].x, fig.data[2].y = za, zb
        fig.data[3].x, fig.data[3].y = [ra], [rb]
        # explicit ranges (incl. the RER point) so the points can't end up outside an un-updated autorange
        for axis, vals, ref in ((fig.layout.xaxis, za, ra), (fig.layout.yaxis, zb, rb)):
            lo, hi = min(vals.min(), ref), max(vals.max(), ref)
            pad = (hi - lo) * 0.08 or abs(hi) * 0.1 or 1.0
            axis.range = [lo - pad, hi + pad]
        fig.layout.annotations[0].text = f"<b>A</b>  {short(w_a.value)}"
        fig.layout.annotations[1].text = f"<b>B</b>  {short(w_b.value)}"
        fig.layout.xaxis.title.text = f"A: {short(w_a.value)} [{ua}]"
        fig.layout.yaxis.title.text = f"B: {short(w_b.value)} [{ub}]"
        fig.layout.title = dict(text=f"{w_proc.value} ({w_scen.value}): per execution", x=0.01, font=dict(size=15))

for w in (w_a, w_b, w_proc, w_scen, w_view): w.observe(update, names="value")
update()
display(widgets.VBox([widgets.HBox([w_a, w_b]), widgets.HBox([w_proc, w_scen]), w_view, fig]))